In [2]:
# Import required libraries
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the SpaceX data
spacex_df = pd.read_csv("spacex_launch_dash.csv")

max_payload = spacex_df["Payload Mass (kg)"].max()
min_payload = spacex_df["Payload Mass (kg)"].min()

# Create Dash app
app = dash.Dash(__name__)

# Dropdown options
site_options = [{"label": "All Sites", "value": "ALL"}]

for site in spacex_df["Launch Site"].unique():
    site_options.append({"label": site, "value": site})

# App layout
app.layout = html.Div(children=[

    html.H1(
        "SpaceX Launch Records Dashboard",
        style={
            "textAlign": "center",
            "color": "#503D36",
            "font-size": 40
        }
    ),

    html.Div([
        dcc.Dropdown(
            id="site-dropdown",
            options=site_options,
            value="ALL",
            placeholder="Select a Launch Site",
            searchable=True
        )
    ]),

    html.Br(),

    html.Div(dcc.Graph(id="success-pie-chart")),

    html.Br(),

    html.P("Payload range (Kg):"),

    dcc.RangeSlider(
        id="payload-slider",
        min=0,
        max=10000,
        step=1000,
        marks={
            0: "0",
            2500: "2500",
            5000: "5000",
            7500: "7500",
            10000: "10000"
        },
        value=[min_payload, max_payload]
    ),

    html.Br(),

    html.Div(dcc.Graph(id="success-payload-scatter-chart"))

])


# Callback for pie chart
@app.callback(
    Output("success-pie-chart", "figure"),
    Input("site-dropdown", "value")
)
def get_pie_chart(selected_site):

    if selected_site == "ALL":
        success_df = spacex_df[spacex_df["class"] == 1]

        fig = px.pie(
            success_df,
            names="Launch Site",
            title="Total Successful Launches by Site"
        )

    else:
        filtered_df = spacex_df[spacex_df["Launch Site"] == selected_site]

        fig = px.pie(
            filtered_df,
            names="class",
            title=f"Success vs Failure for {selected_site}"
        )

    return fig


# Callback for scatter chart
@app.callback(
    Output("success-payload-scatter-chart", "figure"),
    [
        Input("site-dropdown", "value"),
        Input("payload-slider", "value")
    ]
)
def get_scatter_chart(selected_site, payload_range):

    low, high = payload_range

    filtered_df = spacex_df[
        (spacex_df["Payload Mass (kg)"] >= low) &
        (spacex_df["Payload Mass (kg)"] <= high)
    ]

    if selected_site != "ALL":
        filtered_df = filtered_df[
            filtered_df["Launch Site"] == selected_site
        ]

    fig = px.scatter(
        filtered_df,
        x="Payload Mass (kg)",
        y="class",
        color="Booster Version Category",
        title="Payload Mass vs Launch Outcome",
        labels={
            "class": "Launch Success"
        }
    )

    return fig


# Run the app
if __name__ == "__main__":
    app.run(debug=True)